# Crime Layer Preparation

## Objective

Prepare the London borough-level crime dataset for integration into the London Intelligence Dataset.

This notebook evaluates the structure, granularity and data quality of the crime dataset before feature engineering and integration.

In [1]:
import pandas as pd
from pathlib import Path

RAW_DATA = Path("../data/raw")
PROCESSED_DATA = Path("../data/processed")

In [2]:
# Load the current London Intelligence Dataset
# This dataset contains the existing Property and Population layers

london_df = pd.read_csv(
    PROCESSED_DATA / "london_intelligence_dataset.csv"
)

print(london_df.shape)
display(london_df.head())

(264, 14)


,District,Year,Transactions,Average_Price,Median_Price,Min_Price,Max_Price,Price_STD,Average_Price_Growth,Median_Price_Growth,Target_Average_Price_Growth,Target_Median_Price_Growth,Code,Population
0,BARKING AND DAGENHAM,2018,2404,371419.146007,306750.0,100,20495000,9.097848e+05,NaN,NaN,0.059790,0.010595,E09000002,216298.0
1,BARKING AND DAGENHAM,2019,2146,393626.402144,310000.0,100,23160000,1.059141e+06,0.059790,0.010595,0.189852,0.032258,E09000002,218828.0
2,BARKING AND DAGENHAM,2020,1717,468356.971462,320000.0,100,41750000,1.792652e+06,0.189852,0.032258,-0.099888,0.046875,E09000002,219227.0
3,BARKING AND DAGENHAM,2021,2537,421573.503745,335000.0,100,50920000,1.783341e+06,-0.099888,0.046875,0.192617,0.104478,E09000002,218714.0
4,BARKING AND DAGENHAM,2022,2110,502775.688152,370000.0,100,98800000,2.466056e+06,0.192617,0.104478,-0.129364,0.010811,E09000002,220039.0


In [3]:
# Create standard borough key for cross-layer integration
#
# Merge_Key is a technical helper column used only for matching
# different London data layers.
# Original source columns remain unchanged.

london_df["Merge_Key"] = (
    london_df["District"]
    .str.upper()
    .str.strip()
    .replace({
        "WESTMINSTER": "CITY OF WESTMINSTER"
    })
)

print("Merge_Key created successfully.")

display(
    london_df[["District", "Merge_Key"]]
    .drop_duplicates()
    .head(10)
)

Merge_Key created successfully.


,District,Merge_Key
0,BARKING AND DAGENHAM,BARKING AND DAGENHAM
8,BARNET,BARNET
16,BEXLEY,BEXLEY
24,BRENT,BRENT
32,BROMLEY,BROMLEY
40,CAMDEN,CAMDEN
48,CITY OF LONDON,CITY OF LONDON
56,CITY OF WESTMINSTER,CITY OF WESTMINSTER
64,CROYDON,CROYDON
72,EALING,EALING


In [4]:
crime_df = pd.read_csv(
    RAW_DATA / "borough_crime_historical.csv"
)

print(crime_df.shape)

crime_df.head()

(1125, 51)


,Group,SubGroup,BOCU,202007,202008,202009,202010,202011,202012,202101,...,202309,202310,202311,202312,202401,202402,202403,202404,202405,202406
0,ARSON AND CRIMINAL DAMAGE,ARSON,Barking and Dagenham,4,6,2,7,4,2,4,...,4,4,3,4,4,5,6,3,8,2
1,ARSON AND CRIMINAL DAMAGE,CRIMINAL DAMAGE,Barking and Dagenham,122,113,114,120,99,110,100,...,128,82,88,125,126,124,134,112,102,107
2,BURGLARY,BURGLARY BUSINESS AND COMMUNITY,Barking and Dagenham,28,23,32,18,18,24,20,...,38,26,19,20,31,21,30,24,28,33
3,BURGLARY,DOMESTIC BURGLARY,Barking and Dagenham,72,62,55,67,89,92,69,...,0,0,0,0,0,0,0,0,0,0
4,BURGLARY,RES BURGLARY OF A HOME,Barking and Dagenham,0,0,0,0,0,0,0,...,65,47,49,44,57,33,45,42,30,31


In [5]:
print("Shape:", crime_df.shape)

print("\nNumber of Boroughs:")
print(crime_df["BOCU"].nunique())

print("\nNumber of Crime Groups:")
print(crime_df["Group"].nunique())

print("\nNumber of Crime SubGroups:")
print(crime_df["SubGroup"].nunique())

print("\nSample Boroughs:")
print(crime_df["BOCU"].unique()[:10])

print("\nCrime Groups:")
print(crime_df["Group"].unique())

print("\nSample rows:")
display(crime_df.head())

Shape: (1125, 51)

Number of Boroughs:
34

Number of Crime Groups:
13

Number of Crime SubGroups:
35

Sample Boroughs:
['Barking and Dagenham' 'Barnet' 'Bexley' 'Brent' 'Bromley' 'Camden'
 'Croydon' 'Ealing' 'Enfield' 'Greenwich']

Crime Groups:
['ARSON AND CRIMINAL DAMAGE' 'BURGLARY' 'DRUG OFFENCES'
 'FRAUD AND FORGERY' 'MISCELLANEOUS CRIMES AGAINST SOCIETY'
 'POSSESSION OF WEAPONS' 'PUBLIC ORDER OFFENCES' 'ROBBERY'
 'SEXUAL OFFENCES' 'THEFT' 'VEHICLE OFFENCES'
 'VIOLENCE AGAINST THE PERSON' 'NFIB FRAUD']

Sample rows:


,Group,SubGroup,BOCU,202007,202008,202009,202010,202011,202012,202101,...,202309,202310,202311,202312,202401,202402,202403,202404,202405,202406
0,ARSON AND CRIMINAL DAMAGE,ARSON,Barking and Dagenham,4,6,2,7,4,2,4,...,4,4,3,4,4,5,6,3,8,2
1,ARSON AND CRIMINAL DAMAGE,CRIMINAL DAMAGE,Barking and Dagenham,122,113,114,120,99,110,100,...,128,82,88,125,126,124,134,112,102,107
2,BURGLARY,BURGLARY BUSINESS AND COMMUNITY,Barking and Dagenham,28,23,32,18,18,24,20,...,38,26,19,20,31,21,30,24,28,33
3,BURGLARY,DOMESTIC BURGLARY,Barking and Dagenham,72,62,55,67,89,92,69,...,0,0,0,0,0,0,0,0,0,0
4,BURGLARY,RES BURGLARY OF A HOME,Barking and Dagenham,0,0,0,0,0,0,0,...,65,47,49,44,57,33,45,42,30,31


In [6]:
sorted(crime_df["BOCU"].unique())

['Barking and Dagenham',
 'Barnet',
 'Bexley',
 'Brent',
 'Bromley',
 'Camden',
 'Croydon',
 'Ealing',
 'Enfield',
 'Greenwich',
 'Hackney',
 'Hammersmith and Fulham',
 'Haringey',
 'Harrow',
 'Havering',
 'Hillingdon',
 'Hounslow',
 'Islington',
 'Kensington and Chelsea',
 'Kingston upon Thames',
 'Lambeth',
 'Lewisham',
 'London Heathrow and London City Airports',
 'Merton',
 'Newham',
 'Redbridge',
 'Richmond upon Thames',
 'Southwark',
 'Sutton',
 'Tower Hamlets',
 'Unknown',
 'Waltham Forest',
 'Wandsworth',
 'Westminster']

In [7]:
crime_df[crime_df["BOCU"].isin([
    "London Heathrow and London City Airports",
    "Unknown"
])]

,Group,SubGroup,BOCU,202007,202008,202009,202010,202011,202012,202101,...,202309,202310,202311,202312,202401,202402,202403,202404,202405,202406
732,ARSON AND CRIMINAL DAMAGE,ARSON,London Heathrow and London City Airports,0,0,0,0,0,0,0,...,0,0,0,1,3,0,0,0,0,0
733,ARSON AND CRIMINAL DAMAGE,CRIMINAL DAMAGE,London Heathrow and London City Airports,0,0,0,0,0,0,0,...,24,19,17,36,25,27,0,0,0,0
734,BURGLARY,BURGLARY BUSINESS AND COMMUNITY,London Heathrow and London City Airports,0,0,0,0,0,0,0,...,6,3,1,4,2,9,0,0,0,0
735,BURGLARY,RES BURGLARY OF A HOME,London Heathrow and London City Airports,0,0,0,0,0,0,0,...,12,1,7,8,11,7,0,0,0,0
736,BURGLARY,RES BURGLARY OF UNCONNECTED BUILDING,London Heathrow and London City Airports,0,0,0,0,0,0,0,...,2,0,3,3,1,0,0,0,0,0
737,DRUG OFFENCES,POSSESSION OF DRUGS,London Heathrow and London City Airports,0,0,0,0,0,0,0,...,2,8,9,11,5,5,0,0,0,0
738,DRUG OFFENCES,TRAFFICKING OF DRUGS,London Heathrow and London City Airports,0,0,0,0,0,0,0,...,3,3,0,4,3,0,0,0,0,0
739,MISCELLANEOUS CRIMES AGAINST SOCIETY,MISC CRIMES AGAINST SOCIETY,London Heathrow and London City Airports,0,0,0,0,0,0,0,...,114,129,105,103,97,76,0,0,0,0
740,POSSESSION OF WEAPONS,POSSESSION OF WEAPONS,London Heathrow and London City Airports,0,0,0,0,0,0,0,...,0,3,1,2,3,3,0,0,0,0
741,PUBLIC ORDER OFFENCES,OTHER OFFENCES PUBLIC ORDER,London Heathrow and London City Airports,0,0,0,0,0,0,0,...,4,4,0,7,1,7,0,0,0,0


In [8]:
crime_df["BOCU"].value_counts().sort_index()

BOCU
Barking and Dagenham                        33
Barnet                                      33
Bexley                                      32
Brent                                       34
Bromley                                     33
Camden                                      34
Croydon                                     33
Ealing                                      34
Enfield                                     33
Greenwich                                   33
Hackney                                     33
Hammersmith and Fulham                      34
Haringey                                    33
Harrow                                      33
Havering                                    33
Hillingdon                                  34
Hounslow                                    33
Islington                                   34
Kensington and Chelsea                      33
Kingston upon Thames                        33
Lambeth                                     34
Lewisham

In [9]:
len(sorted(crime_df["BOCU"].unique()))

34

In [10]:
# Remove non-borough records and validate 33 London boroughs
crime_clean = crime_df[
    ~crime_df["BOCU"].isin([
        "London Heathrow and London City Airports",
        "Unknown"
    ])
]

print("Unique Boroughs:", crime_clean["BOCU"].nunique())
print(sorted(crime_clean["BOCU"].unique()))

Unique Boroughs: 32
['Barking and Dagenham', 'Barnet', 'Bexley', 'Brent', 'Bromley', 'Camden', 'Croydon', 'Ealing', 'Enfield', 'Greenwich', 'Hackney', 'Hammersmith and Fulham', 'Haringey', 'Harrow', 'Havering', 'Hillingdon', 'Hounslow', 'Islington', 'Kensington and Chelsea', 'Kingston upon Thames', 'Lambeth', 'Lewisham', 'Merton', 'Newham', 'Redbridge', 'Richmond upon Thames', 'Southwark', 'Sutton', 'Tower Hamlets', 'Waltham Forest', 'Wandsworth', 'Westminster']


In [11]:
month_cols = crime_clean.columns[3:]

print(month_cols[0])
print(month_cols[-1])
print(len(month_cols))

202007
202406
48


In [12]:
# Convert monthly crime data from wide to long format

crime_long = crime_clean.melt(
    id_vars=["BOCU", "Group", "SubGroup"],
    var_name="YearMonth",
    value_name="Crime_Count"
)

print("Shape:", crime_long.shape)
display(crime_long.head())

Shape: (51120, 5)


,BOCU,Group,SubGroup,YearMonth,Crime_Count
0,Barking and Dagenham,ARSON AND CRIMINAL DAMAGE,ARSON,202007,4
1,Barking and Dagenham,ARSON AND CRIMINAL DAMAGE,CRIMINAL DAMAGE,202007,122
2,Barking and Dagenham,BURGLARY,BURGLARY BUSINESS AND COMMUNITY,202007,28
3,Barking and Dagenham,BURGLARY,DOMESTIC BURGLARY,202007,72
4,Barking and Dagenham,BURGLARY,RES BURGLARY OF A HOME,202007,0


In [13]:
# Convert YYYYMM into a proper monthly date and extract Year/Month for analysis

crime_long["Date"] = pd.to_datetime(
    crime_long["YearMonth"].astype(str),
    format="%Y%m"
)

crime_long["Year"] = crime_long["Date"].dt.year
crime_long["Month"] = crime_long["Date"].dt.month

display(crime_long.head())
print(crime_long[["Year", "Month"]].drop_duplicates().sort_values(["Year", "Month"]).head(12))

,BOCU,Group,SubGroup,YearMonth,Crime_Count,Date,Year,Month
0,Barking and Dagenham,ARSON AND CRIMINAL DAMAGE,ARSON,202007,4,2020-07-01,2020,7
1,Barking and Dagenham,ARSON AND CRIMINAL DAMAGE,CRIMINAL DAMAGE,202007,122,2020-07-01,2020,7
2,Barking and Dagenham,BURGLARY,BURGLARY BUSINESS AND COMMUNITY,202007,28,2020-07-01,2020,7
3,Barking and Dagenham,BURGLARY,DOMESTIC BURGLARY,202007,72,2020-07-01,2020,7
4,Barking and Dagenham,BURGLARY,RES BURGLARY OF A HOME,202007,0,2020-07-01,2020,7


       Year  Month
0      2020      7
1065   2020      8
2130   2020      9
3195   2020     10
4260   2020     11
5325   2020     12
6390   2021      1
7455   2021      2
8520   2021      3
9585   2021      4
10650  2021      5
11715  2021      6


## Filter to Complete Calendar Years

The crime dataset covers **July 2020 to June 2024**, so the first and last years are incomplete.

To make annual comparisons consistent, only **complete calendar years (2021–2023)** are retained for feature engineering.

This ensures that yearly crime statistics are calculated from full years before being merged into the London Intelligence Dataset.

In [14]:
# Retain only complete calendar years (2021–2023)

crime_complete_years = crime_long[
    crime_long["Year"].between(2021, 2023)
].copy()

print("Shape:", crime_complete_years.shape)
print("\nYears:")
print(sorted(crime_complete_years["Year"].unique()))

Shape: (38340, 8)

Years:
[np.int32(2021), np.int32(2022), np.int32(2023)]


## Aggregate Crime by Group (Monthly)

At this stage, crime counts are aggregated from **SubGroup** level to **Group** level for each borough and month.

This creates a cleaner monthly dataset while preserving the main crime categories for later annual feature engineering.

In [15]:
# Aggregate monthly crime counts from SubGroup to Group level

crime_group_monthly = (
    crime_complete_years
    .groupby(
        ["BOCU", "Group", "Date", "Year", "Month"],
        as_index=False
    )["Crime_Count"]
    .sum()
)

print("Shape:", crime_group_monthly.shape)
display(crime_group_monthly.head())

Shape: (14184, 6)


,BOCU,Group,Date,Year,Month,Crime_Count
0,Barking and Dagenham,ARSON AND CRIMINAL DAMAGE,2021-01-01,2021,1,104
1,Barking and Dagenham,ARSON AND CRIMINAL DAMAGE,2021-02-01,2021,2,109
2,Barking and Dagenham,ARSON AND CRIMINAL DAMAGE,2021-03-01,2021,3,86
3,Barking and Dagenham,ARSON AND CRIMINAL DAMAGE,2021-04-01,2021,4,106
4,Barking and Dagenham,ARSON AND CRIMINAL DAMAGE,2021-05-01,2021,5,127


In [16]:
# Validate the number of monthly records per borough and crime group

group_counts = (
    crime_group_monthly
    .groupby(["BOCU", "Group"])
    .size()
    .reset_index(name="Months")
)

print(group_counts["Months"].value_counts().sort_index())

display(group_counts.head())

Months
36    394
Name: count, dtype: int64


,BOCU,Group,Months
0,Barking and Dagenham,ARSON AND CRIMINAL DAMAGE,36
1,Barking and Dagenham,BURGLARY,36
2,Barking and Dagenham,DRUG OFFENCES,36
3,Barking and Dagenham,FRAUD AND FORGERY,36
4,Barking and Dagenham,MISCELLANEOUS CRIMES AGAINST SOCIETY,36


## Aggregate Crime by Group (Yearly)

Monthly crime counts are aggregated to annual totals for each borough and crime group.

This creates yearly crime statistics that can later be transformed into borough-level features and merged into the London Intelligence Dataset.

In [17]:
# Aggregate monthly crime counts to yearly totals

crime_group_yearly = (
    crime_group_monthly
    .groupby(
        ["BOCU", "Year", "Group"],
        as_index=False
    )["Crime_Count"]
    .sum()
)

print("Shape:", crime_group_yearly.shape)
display(crime_group_yearly.head())

Shape: (1182, 4)


,BOCU,Year,Group,Crime_Count
0,Barking and Dagenham,2021,ARSON AND CRIMINAL DAMAGE,1424
1,Barking and Dagenham,2021,BURGLARY,1163
2,Barking and Dagenham,2021,DRUG OFFENCES,1463
3,Barking and Dagenham,2021,FRAUD AND FORGERY,0
4,Barking and Dagenham,2021,MISCELLANEOUS CRIMES AGAINST SOCIETY,315


In [18]:
print("Shape:", crime_group_yearly.shape)

print("\nYears:")
print(sorted(crime_group_yearly["Year"].unique()))

print("\nCrime Groups:")
print(crime_group_yearly["Group"].nunique())

display(crime_group_yearly.head())

Shape: (1182, 4)

Years:
[np.int32(2021), np.int32(2022), np.int32(2023)]

Crime Groups:
13


,BOCU,Year,Group,Crime_Count
0,Barking and Dagenham,2021,ARSON AND CRIMINAL DAMAGE,1424
1,Barking and Dagenham,2021,BURGLARY,1163
2,Barking and Dagenham,2021,DRUG OFFENCES,1463
3,Barking and Dagenham,2021,FRAUD AND FORGERY,0
4,Barking and Dagenham,2021,MISCELLANEOUS CRIMES AGAINST SOCIETY,315


## Create Borough-Year Crime Features

The yearly crime dataset is reshaped from a long format into a wide format.

Each crime group becomes a separate feature, producing one record per borough and year.

Missing borough-crime group combinations are filled with zero, as the absence of a record represents no recorded incidents rather than missing information.

This feature table will be merged into the London Intelligence Dataset.

In [19]:
# Pivot yearly crime groups into borough-year features

crime_features = (
    crime_group_yearly
    .pivot_table(
        index=["BOCU", "Year"],
        columns="Group",
        values="Crime_Count",
        fill_value=0
    )
    .reset_index()
)
# Remove the column index name created by pivot
crime_features.columns.name = None

print("Shape:", crime_features.shape)

display(crime_features.head())

Shape: (96, 15)


,BOCU,Year,ARSON AND CRIMINAL DAMAGE,BURGLARY,DRUG OFFENCES,FRAUD AND FORGERY,MISCELLANEOUS CRIMES AGAINST SOCIETY,NFIB FRAUD,POSSESSION OF WEAPONS,PUBLIC ORDER OFFENCES,ROBBERY,SEXUAL OFFENCES,THEFT,VEHICLE OFFENCES,VIOLENCE AGAINST THE PERSON
0,Barking and Dagenham,2021,1424.0,1163.0,1463.0,0.0,315.0,0.0,161.0,1257.0,546.0,514.0,3085.0,2324.0,6406.0
1,Barking and Dagenham,2022,1431.0,1100.0,1417.0,0.0,322.0,0.0,158.0,1245.0,595.0,626.0,3502.0,2789.0,6581.0
2,Barking and Dagenham,2023,1333.0,1077.0,1343.0,1.0,274.0,0.0,210.0,1359.0,863.0,586.0,4356.0,2640.0,7013.0
3,Barnet,2021,1780.0,2339.0,1076.0,0.0,395.0,0.0,144.0,2062.0,605.0,726.0,4959.0,4893.0,7832.0
4,Barnet,2022,1805.0,2519.0,993.0,0.0,374.0,0.0,157.0,1862.0,720.0,737.0,5620.0,4954.0,7684.0


## Validate Crime Categories

Before merging the Crime Layer, we verify that crime categories are consistently recorded across all years.

This validation helps identify structural changes in the source data, such as renamed or newly introduced crime categories.

In [20]:
# Check yearly availability of each crime group

crime_group_summary = (
    crime_group_yearly
    .pivot_table(
        index="Group",
        columns="Year",
        values="Crime_Count",
        aggfunc="sum"
    )
)

display(crime_group_summary)

Year,2021,2022,2023
Group,,,
ARSON AND CRIMINAL DAMAGE,51801,51973,55937
BURGLARY,52337,51685,55369
DRUG OFFENCES,45446,43485,38543
FRAUD AND FORGERY,4,6,8
MISCELLANEOUS CRIMES AGAINST SOCIETY,11185,11806,10299
NFIB FRAUD,0,0,0
POSSESSION OF WEAPONS,5995,6309,6365
PUBLIC ORDER OFFENCES,56642,55012,57839
ROBBERY,22022,26386,32674


## Validate Constant Crime Features

Features with the same value across all observations provide no useful information for analysis or predictive modelling.

This validation identifies crime categories with no variation before the final feature set is created.

In [21]:
# Check whether any crime feature has a constant value

crime_feature_variation = crime_features.nunique().sort_values()

print(crime_feature_variation)

NFIB FRAUD                               1
Year                                     3
FRAUD AND FORGERY                        4
BOCU                                    32
POSSESSION OF WEAPONS                   79
MISCELLANEOUS CRIMES AGAINST SOCIETY    86
ROBBERY                                 90
SEXUAL OFFENCES                         92
DRUG OFFENCES                           93
BURGLARY                                94
PUBLIC ORDER OFFENCES                   94
THEFT                                   95
VEHICLE OFFENCES                        95
ARSON AND CRIMINAL DAMAGE               96
VIOLENCE AGAINST THE PERSON             96
dtype: int64


## Remove Constant Features

The `NFIB FRAUD` feature contains the same value for every borough-year record during the analysis period (2021–2023).

Since constant features provide no useful information for analysis or predictive modelling, this feature is removed from the final Crime Layer.

In [22]:
# Remove constant crime features

crime_features = crime_features.drop(
    columns=["NFIB FRAUD"]
)

print(crime_features.shape)

display(crime_features.head())

(96, 14)


,BOCU,Year,ARSON AND CRIMINAL DAMAGE,BURGLARY,DRUG OFFENCES,FRAUD AND FORGERY,MISCELLANEOUS CRIMES AGAINST SOCIETY,POSSESSION OF WEAPONS,PUBLIC ORDER OFFENCES,ROBBERY,SEXUAL OFFENCES,THEFT,VEHICLE OFFENCES,VIOLENCE AGAINST THE PERSON
0,Barking and Dagenham,2021,1424.0,1163.0,1463.0,0.0,315.0,161.0,1257.0,546.0,514.0,3085.0,2324.0,6406.0
1,Barking and Dagenham,2022,1431.0,1100.0,1417.0,0.0,322.0,158.0,1245.0,595.0,626.0,3502.0,2789.0,6581.0
2,Barking and Dagenham,2023,1333.0,1077.0,1343.0,1.0,274.0,210.0,1359.0,863.0,586.0,4356.0,2640.0,7013.0
3,Barnet,2021,1780.0,2339.0,1076.0,0.0,395.0,144.0,2062.0,605.0,726.0,4959.0,4893.0,7832.0
4,Barnet,2022,1805.0,2519.0,993.0,0.0,374.0,157.0,1862.0,720.0,737.0,5620.0,4954.0,7684.0


## Pre-Merge Validation

Before merging the Crime Layer into the London Intelligence Dataset, both datasets are validated to ensure they share the same grain and compatible merge keys.

The validation checks:

- Number of records
- Years
- Duplicate borough-year records
- Borough coverage

In [23]:
# -----------------------------
# Pre-Merge Validation
# -----------------------------

print("=" * 50)
print("London Intelligence Dataset")
print("=" * 50)

print("Shape:", london_df.shape)
print("Years:", sorted(london_df["Year"].unique()))

print("\nDuplicate District-Year records:",
      london_df.duplicated(subset=["District", "Year"]).sum())


print("\n" + "=" * 50)
print("Crime Features")
print("=" * 50)

print("Shape:", crime_features.shape)
print("Years:", sorted(crime_features["Year"].unique()))

print("\nDuplicate Borough-Year records:",
      crime_features.duplicated(subset=["BOCU", "Year"]).sum())

London Intelligence Dataset
Shape: (264, 15)
Years: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Duplicate District-Year records: 0

Crime Features
Shape: (96, 14)
Years: [np.int32(2021), np.int32(2022), np.int32(2023)]

Duplicate Borough-Year records: 0


In [24]:
# -----------------------------
# Compare raw borough names
# -----------------------------
#
# Compare the original borough names from the Property and Crime datasets
# before standardisation.
#
# This validation demonstrates why a standardised Merge_Key is required
# for reliable cross-layer integration.

property_boroughs = set(london_df["District"])
crime_boroughs = set(crime_features["BOCU"])

print("Only in Crime:")
print(sorted(crime_boroughs - property_boroughs))

print("\nOnly in Property:")
print(sorted(property_boroughs - crime_boroughs))

Only in Crime:
['Barking and Dagenham', 'Barnet', 'Bexley', 'Brent', 'Bromley', 'Camden', 'Croydon', 'Ealing', 'Enfield', 'Greenwich', 'Hackney', 'Hammersmith and Fulham', 'Haringey', 'Harrow', 'Havering', 'Hillingdon', 'Hounslow', 'Islington', 'Kensington and Chelsea', 'Kingston upon Thames', 'Lambeth', 'Lewisham', 'Merton', 'Newham', 'Redbridge', 'Richmond upon Thames', 'Southwark', 'Sutton', 'Tower Hamlets', 'Waltham Forest', 'Wandsworth', 'Westminster']

Only in Property:
['BARKING AND DAGENHAM', 'BARNET', 'BEXLEY', 'BRENT', 'BROMLEY', 'CAMDEN', 'CITY OF LONDON', 'CITY OF WESTMINSTER', 'CROYDON', 'EALING', 'ENFIELD', 'GREENWICH', 'HACKNEY', 'HAMMERSMITH AND FULHAM', 'HARINGEY', 'HARROW', 'HAVERING', 'HILLINGDON', 'HOUNSLOW', 'ISLINGTON', 'KENSINGTON AND CHELSEA', 'KINGSTON UPON THAMES', 'LAMBETH', 'LEWISHAM', 'MERTON', 'NEWHAM', 'REDBRIDGE', 'RICHMOND UPON THAMES', 'SOUTHWARK', 'SUTTON', 'TOWER HAMLETS', 'WALTHAM FOREST', 'WANDSWORTH']


## Create Merge Key

The original `BOCU` names are preserved to maintain the integrity of the source data.

A separate `Merge_Key` is created and standardised for dataset integration.

This approach provides a consistent merge strategy across all external data layers while keeping the original source values unchanged.

In [25]:
# Create a standardised merge key

crime_features["Merge_Key"] = (
    crime_features["BOCU"]
    .str.upper()
    .replace({
        "WESTMINSTER": "CITY OF WESTMINSTER"
    })
)

display(
    crime_features[
        ["BOCU", "Merge_Key"]
    ].drop_duplicates().sort_values("BOCU")
)

,BOCU,Merge_Key
0,Barking and Dagenham,BARKING AND DAGENHAM
3,Barnet,BARNET
6,Bexley,BEXLEY
9,Brent,BRENT
12,Bromley,BROMLEY
15,Camden,CAMDEN
18,Croydon,CROYDON
21,Ealing,EALING
24,Enfield,ENFIELD
27,Greenwich,GREENWICH


In [26]:
property_keys = set(london_df["Merge_Key"])
crime_keys = set(crime_features["Merge_Key"])

print("Only in Crime:")
print(sorted(crime_keys - property_keys))

print("\nOnly in Property:")
print(sorted(property_keys - crime_keys))

Only in Crime:
[]

Only in Property:
['CITY OF LONDON']


In [27]:
print(london_df.columns.tolist())

['District', 'Year', 'Transactions', 'Average_Price', 'Median_Price', 'Min_Price', 'Max_Price', 'Price_STD', 'Average_Price_Growth', 'Median_Price_Growth', 'Target_Average_Price_Growth', 'Target_Median_Price_Growth', 'Code', 'Population', 'Merge_Key']


## Final Audit Before Merge
Before merging the Crime Layer into the London Intelligence Dataset, a final audit is performed.

This audit verifies that the engineered crime features are complete, uniquely identified, free from duplicate records, and ready for integration.

Performing this validation before every merge helps maintain data quality throughout the project.

In [28]:
# -----------------------------
# Audit Crime Feature Table
# -----------------------------
#
# Perform a final quality check before merging the Crime Layer.
#
# Validation includes:
# - Dataset shape
# - Year coverage
# - Borough coverage
# - Duplicate Merge_Key-Year records
# - Missing values
# - Data types

print("=" * 60)
print("Crime Features Audit")
print("=" * 60)

print(f"Shape: {crime_features.shape}")

print("\nYears:")
print(sorted(crime_features["Year"].unique()))

print("\nUnique Boroughs:")
print(crime_features["Merge_Key"].nunique())

print("\nDuplicate Merge_Key-Year records:")
print(
    crime_features.duplicated(
        subset=["Merge_Key", "Year"]
    ).sum()
)

print("\nMissing Values:")
print(crime_features.isna().sum())

print("\nData Types:")
print(crime_features.dtypes)

Crime Features Audit
Shape: (96, 15)

Years:
[np.int32(2021), np.int32(2022), np.int32(2023)]

Unique Boroughs:
32

Duplicate Merge_Key-Year records:
0

Missing Values:
BOCU                                    0
Year                                    0
ARSON AND CRIMINAL DAMAGE               0
BURGLARY                                0
DRUG OFFENCES                           0
FRAUD AND FORGERY                       0
MISCELLANEOUS CRIMES AGAINST SOCIETY    0
POSSESSION OF WEAPONS                   0
PUBLIC ORDER OFFENCES                   0
ROBBERY                                 0
SEXUAL OFFENCES                         0
THEFT                                   0
VEHICLE OFFENCES                        0
VIOLENCE AGAINST THE PERSON             0
Merge_Key                               0
dtype: int64

Data Types:
BOCU                                     object
Year                                      int32
ARSON AND CRIMINAL DAMAGE               float64
BURGLARY                       

In [29]:
# ============================================================
# Export Crime Feature Layer
# ============================================================

CRIME_OUTPUT_PATH = (
    PROCESSED_DATA /
    "london_crime.csv"
)

crime_features.to_csv(
    CRIME_OUTPUT_PATH,
    index=False
)

print("Crime feature layer saved successfully.")
print("Output path:", CRIME_OUTPUT_PATH)
print("Shape:", crime_features.shape)


Crime feature layer saved successfully.
Output path: ..\data\processed\london_crime.csv
Shape: (96, 15)


In [30]:
# ============================================================
# Validate Exported Crime Feature Layer
# ============================================================

crime_check = pd.read_csv(
    CRIME_OUTPUT_PATH
)

print("=" * 60)
print("EXPORTED CRIME LAYER VALIDATION")
print("=" * 60)

print("Shape:", crime_check.shape)

print("\nColumns:")
print(crime_check.columns.tolist())

print("\nYears:")
print(sorted(crime_check["Year"].unique()))

print("\nUnique Boroughs:")
print(crime_check["Merge_Key"].nunique())

print("\nDuplicate Borough-Year records:")
print(
    crime_check.duplicated(
        subset=["Merge_Key", "Year"]
    ).sum()
)

print("\nMissing values:")
print(crime_check.isna().sum())


EXPORTED CRIME LAYER VALIDATION
Shape: (96, 15)

Columns:
['BOCU', 'Year', 'ARSON AND CRIMINAL DAMAGE', 'BURGLARY', 'DRUG OFFENCES', 'FRAUD AND FORGERY', 'MISCELLANEOUS CRIMES AGAINST SOCIETY', 'POSSESSION OF WEAPONS', 'PUBLIC ORDER OFFENCES', 'ROBBERY', 'SEXUAL OFFENCES', 'THEFT', 'VEHICLE OFFENCES', 'VIOLENCE AGAINST THE PERSON', 'Merge_Key']

Years:
[np.int64(2021), np.int64(2022), np.int64(2023)]

Unique Boroughs:
32

Duplicate Borough-Year records:
0

Missing values:
BOCU                                    0
Year                                    0
ARSON AND CRIMINAL DAMAGE               0
BURGLARY                                0
DRUG OFFENCES                           0
FRAUD AND FORGERY                       0
MISCELLANEOUS CRIMES AGAINST SOCIETY    0
POSSESSION OF WEAPONS                   0
PUBLIC ORDER OFFENCES                   0
ROBBERY                                 0
SEXUAL OFFENCES                         0
THEFT                                   0
VEHICLE OFFENCES   

## Merge Crime Layer

The Crime Layer is merged into the London Intelligence Dataset using the standard `Merge_Key` and `Year`.

A left join is used to preserve all existing borough-year records in the London Intelligence Dataset while adding crime features where available.

Since crime data is available only for 2021–2023, records outside this period will naturally contain missing values for the newly added crime features.

In [31]:
# Merge Crime Layer into the London Intelligence Dataset

london_df = london_df.merge(
    crime_features.drop(columns=["BOCU"]),
    on=["Merge_Key", "Year"],
    how="left"
)

print("Shape:", london_df.shape)

display(london_df.head())

Shape: (264, 27)


,District,Year,Transactions,Average_Price,Median_Price,Min_Price,Max_Price,Price_STD,Average_Price_Growth,Median_Price_Growth,...,DRUG OFFENCES,FRAUD AND FORGERY,MISCELLANEOUS CRIMES AGAINST SOCIETY,POSSESSION OF WEAPONS,PUBLIC ORDER OFFENCES,ROBBERY,SEXUAL OFFENCES,THEFT,VEHICLE OFFENCES,VIOLENCE AGAINST THE PERSON
0,BARKING AND DAGENHAM,2018,2404,371419.146007,306750.0,100,20495000,9.097848e+05,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,BARKING AND DAGENHAM,2019,2146,393626.402144,310000.0,100,23160000,1.059141e+06,0.059790,0.010595,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,BARKING AND DAGENHAM,2020,1717,468356.971462,320000.0,100,41750000,1.792652e+06,0.189852,0.032258,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BARKING AND DAGENHAM,2021,2537,421573.503745,335000.0,100,50920000,1.783341e+06,-0.099888,0.046875,...,1463.0,0.0,315.0,161.0,1257.0,546.0,514.0,3085.0,2324.0,6406.0
4,BARKING AND DAGENHAM,2022,2110,502775.688152,370000.0,100,98800000,2.466056e+06,0.192617,0.104478,...,1417.0,0.0,322.0,158.0,1245.0,595.0,626.0,3502.0,2789.0,6581.0


## Validate Merge Result

After merging the Crime Layer, the integrated dataset is validated to ensure:

- The number of records is unchanged.
- No duplicate District-Year records were introduced.
- Crime features are available only for the expected years (2021–2023).
- Borough coverage matches the available crime data.
- The merge completed successfully before saving the updated dataset.

In [32]:
# -----------------------------
# Validate Merge Result
# -----------------------------

print("=" * 60)
print("Merged Dataset Validation")
print("=" * 60)

print(f"Shape: {london_df.shape}")

print("\nYears:")
print(sorted(london_df["Year"].unique()))

print("\nUnique Districts:")
print(london_df["District"].nunique())

print("\nDuplicate District-Year records:")
print(
    london_df.duplicated(
        subset=["District", "Year"]
    ).sum()
)

print("\nCrime feature coverage by Year:")

coverage = (
    london_df
    .groupby("Year")["THEFT"]
    .apply(lambda x: x.notna().sum())
)

print(coverage)

Merged Dataset Validation
Shape: (264, 27)

Years:
[np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Unique Districts:
33

Duplicate District-Year records:
0

Crime feature coverage by Year:
Year
2018     0
2019     0
2020     0
2021    32
2022    32
2023    32
2024     0
2025     0
Name: THEFT, dtype: int64


### Validation Summary

The Crime Layer integration has been successfully validated.

The validation checks confirm that:

- The merged dataset preserves all **264 borough-year records** from the original London Intelligence Dataset, with no records added, removed, or duplicated during the merge process.
- No duplicate **District-Year** combinations were introduced, confirming that the dataset grain remains consistent.
- Crime features are populated only for **2021–2023**, which aligns with the available time period of the source crime dataset.
- Crime data coverage includes **32 London boroughs** for each available year. **City of London** contains missing crime values because it is not represented in the original crime data source.

Based on these checks, the merged dataset maintains structural integrity and is ready for the final audit and export stage.

In [33]:
# ============================================================
# Final Dataset Audit
# ============================================================
#
# Final quality checks after integrating the Crime Layer.
#
# Validation includes:
# - Final dataset shape
# - Column structure
# - Year coverage
# - Borough coverage
# - Duplicate records
# - Missing values
# - Data types
# ============================================================


print("=" * 60)
print("Final Dataset Audit")
print("=" * 60)


print("\nDataset Shape:")
print(london_df.shape)


print("\nYears:")
print(sorted(london_df["Year"].unique()))


print("\nNumber of Boroughs:")
print(london_df["Merge_Key"].nunique())


print("\nDuplicate District-Year records:")
print(
    london_df.duplicated(
        subset=["District", "Year"]
    ).sum()
)


print("\nMissing Values:")
display(
    london_df.isna().sum()
)


print("\nData Types:")
display(
    london_df.dtypes
)


print("\nColumns:")
print(london_df.columns.tolist())

Final Dataset Audit

Dataset Shape:
(264, 27)

Years:
[np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Number of Boroughs:
33

Duplicate District-Year records:
0

Missing Values:


District                                  0
Year                                      0
Transactions                              0
Average_Price                             0
Median_Price                              0
Min_Price                                 0
Max_Price                                 0
Price_STD                                 0
Average_Price_Growth                     33
Median_Price_Growth                      33
Target_Average_Price_Growth              33
Target_Median_Price_Growth               33
Code                                     99
Population                               99
Merge_Key                                 0
ARSON AND CRIMINAL DAMAGE               168
BURGLARY                                168
DRUG OFFENCES                           168
FRAUD AND FORGERY                       168
MISCELLANEOUS CRIMES AGAINST SOCIETY    168
POSSESSION OF WEAPONS                   168
PUBLIC ORDER OFFENCES                   168
ROBBERY                         


Data Types:


District                                 object
Year                                      int64
Transactions                              int64
Average_Price                           float64
Median_Price                            float64
Min_Price                                 int64
Max_Price                                 int64
Price_STD                               float64
Average_Price_Growth                    float64
Median_Price_Growth                     float64
Target_Average_Price_Growth             float64
Target_Median_Price_Growth              float64
Code                                     object
Population                              float64
Merge_Key                                object
ARSON AND CRIMINAL DAMAGE               float64
BURGLARY                                float64
DRUG OFFENCES                           float64
FRAUD AND FORGERY                       float64
MISCELLANEOUS CRIMES AGAINST SOCIETY    float64
POSSESSION OF WEAPONS                   


Columns:
['District', 'Year', 'Transactions', 'Average_Price', 'Median_Price', 'Min_Price', 'Max_Price', 'Price_STD', 'Average_Price_Growth', 'Median_Price_Growth', 'Target_Average_Price_Growth', 'Target_Median_Price_Growth', 'Code', 'Population', 'Merge_Key', 'ARSON AND CRIMINAL DAMAGE', 'BURGLARY', 'DRUG OFFENCES', 'FRAUD AND FORGERY', 'MISCELLANEOUS CRIMES AGAINST SOCIETY', 'POSSESSION OF WEAPONS', 'PUBLIC ORDER OFFENCES', 'ROBBERY', 'SEXUAL OFFENCES', 'THEFT', 'VEHICLE OFFENCES', 'VIOLENCE AGAINST THE PERSON']


## Final Dataset Audit Summary

The final London Intelligence Dataset successfully integrates the Crime Layer with the existing Property and Population layers.

The final validation confirms that:

- The dataset contains 264 borough-year observations with 27 features.
- No duplicate borough-year records were introduced during integration.
- Borough coverage remains complete across 33 London boroughs.
- Crime features are available for the source coverage period (2021–2023).
- Missing crime values outside this period are expected due to limited crime data availability.
- City of London remains without crime features because it is not included in the source crime dataset.
- Missing growth values correspond to the first observation year where historical comparison is unavailable.

The dataset is structurally valid and ready for export.

In [34]:
# ============================================================
# Save Final London Intelligence Dataset
# ============================================================
#
# Export the final integrated dataset containing:
# - Property layer
# - Population layer
# - Crime layer
#
# The exported file will be used as the main processed dataset
# for downstream analysis and modelling.
# ============================================================


FINAL_DATA_PATH = (
    PROCESSED_DATA /
    "london_intelligence_dataset_with_crime.csv"
)


london_df.to_csv(
    FINAL_DATA_PATH,
    index=False
)


print("Dataset saved successfully:")
print(FINAL_DATA_PATH)

Dataset saved successfully:
..\data\processed\london_intelligence_dataset_with_crime.csv


In [35]:
# ============================================================
# Verify Saved Dataset
# ============================================================

check_df = pd.read_csv(
    FINAL_DATA_PATH
)


print("Reloaded dataset shape:")
print(check_df.shape)


display(check_df.head())

Reloaded dataset shape:
(264, 27)


,District,Year,Transactions,Average_Price,Median_Price,Min_Price,Max_Price,Price_STD,Average_Price_Growth,Median_Price_Growth,...,DRUG OFFENCES,FRAUD AND FORGERY,MISCELLANEOUS CRIMES AGAINST SOCIETY,POSSESSION OF WEAPONS,PUBLIC ORDER OFFENCES,ROBBERY,SEXUAL OFFENCES,THEFT,VEHICLE OFFENCES,VIOLENCE AGAINST THE PERSON
0,BARKING AND DAGENHAM,2018,2404,371419.146007,306750.0,100,20495000,9.097848e+05,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,BARKING AND DAGENHAM,2019,2146,393626.402144,310000.0,100,23160000,1.059141e+06,0.059790,0.010595,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,BARKING AND DAGENHAM,2020,1717,468356.971462,320000.0,100,41750000,1.792652e+06,0.189852,0.032258,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BARKING AND DAGENHAM,2021,2537,421573.503745,335000.0,100,50920000,1.783341e+06,-0.099888,0.046875,...,1463.0,0.0,315.0,161.0,1257.0,546.0,514.0,3085.0,2324.0,6406.0
4,BARKING AND DAGENHAM,2022,2110,502775.688152,370000.0,100,98800000,2.466056e+06,0.192617,0.104478,...,1417.0,0.0,322.0,158.0,1245.0,595.0,626.0,3502.0,2789.0,6581.0


In [36]:
print("Years:")
print(sorted(london_df["Year"].unique()))

Years:
[np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
